# Gradient boosting do scikit-learn (linha extra)

**Estado:** o que está executado aqui é o trial 0 e um estudo curto de 5 trials, só para provar que o fluxo funciona.
O N de trials ainda não foi definido e o conjunto de teste não é usado neste notebook.
O estudo curto fica em `resultados/curto/`, fora do Git, para o estudo completo não retomar a partir dele.

Modelo, faixas e trial 0 seguem a Frente 4 de `planos/rafael.md`.
A referência da comparação é o XGBoost.
O treino cresce com `warm_start`, uma árvore por vez, e para depois de 10 árvores sem queda do log loss na validação da dobra, a mesma regra do XGBoost.
O scikit-learn não tem MSE como perda deste classificador, então o modelo não entra na pergunta "entropia cruzada ou MSE" e fica com `por_perda` vazio.

In [1]:
import sys

sys.path.insert(0, "..")

import platform
import time
from itertools import islice

import numpy as np
import optuna
import sklearn
from sklearn.ensemble import GradientBoostingClassifier

import comum

print(sys.executable)
print("Python", platform.python_version(), "|", "scikit-learn", sklearn.__version__, "| optuna", optuna.__version__)

C:\dev\Faculdade\churn_project\.venv\Scripts\python.exe
Python 3.12.10 | scikit-learn 1.9.1 | optuna 5.0.0


In [2]:
MODELO = "gradient_boosting"
N_TRIALS = 5  # parâmetro: o N final ainda não foi definido
ESTUDO_CURTO = True  # False grava em resultados/gradient_boosting.db, que é o estudo que conta
TETO_ARVORES = 10_000
PACIENCIA = 10

# O teste, que vem em `final`, fica de fora de propósito.
folds, _, colunas = comum.carregar_dobras()
print(len(folds), "dobras,", len(colunas), "colunas")

5 dobras, 28 colunas


In [3]:
PADRAO = dict(
    loss="log_loss",
    learning_rate=0.1,
    max_depth=3,
    min_samples_split=2,
    min_samples_leaf=1,
    subsample=1.0,
    max_features=1.0,
    min_impurity_decrease=0.0,
)


def sugerir(t):
    return dict(
        loss=t.suggest_categorical("loss", ["log_loss", "exponential"]),
        learning_rate=t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_depth=t.suggest_int("max_depth", 2, 10),
        min_samples_split=t.suggest_int("min_samples_split", 2, 50),
        min_samples_leaf=t.suggest_int("min_samples_leaf", 1, 50),
        subsample=t.suggest_float("subsample", 0.5, 1),
        max_features=t.suggest_float("max_features", 0.3, 1),
        min_impurity_decrease=t.suggest_float("min_impurity_decrease", 0, 0.05),
    )


def ajustador(hp, arvores):
    """Devolve o `ajustar` de comum.avaliar_cv; anota em `arvores` quantas cada dobra usou."""

    def ajustar(dobra):
        Xt, yt, Xv, yv = dobra
        modelo = GradientBoostingClassifier(**hp, warm_start=True, random_state=comum.SEMENTE)
        melhor_perda, melhor_n, sem_queda = np.inf, 0, 0
        for n in range(1, TETO_ARVORES + 1):
            modelo.set_params(n_estimators=n).fit(Xt, yt)
            perda = comum.ce(yv, modelo.predict_proba(Xv)[:, 1])
            if perda < melhor_perda:
                melhor_perda, melhor_n, sem_queda = perda, n, 0
            else:
                sem_queda += 1
                if sem_queda >= PACIENCIA:
                    break
        arvores.append(melhor_n)

        def prever(X):  # só as árvores até a melhor
            return next(islice(modelo.staged_predict_proba(X), melhor_n - 1, None))[:, 1]

        return prever

    return ajustar


def objetivo(trial):
    arvores = []
    r = comum.avaliar_cv(ajustador(sugerir(trial), arvores), folds, trial)
    trial.set_user_attr("arvores_dobras", arvores)
    return r["ks"]

## Trial 0

Configuração padrão do scikit-learn, fora do estudo, para conferir contra a medida do plano.

In [4]:
arvores0 = []
t0 = time.perf_counter()
r0 = comum.avaliar_cv(ajustador(PADRAO, arvores0), folds)
print(f"KS médio {r0['ks']:.4f} | por dobra {np.round(r0['ks_dobras'], 4)}")
print(f"árvores por dobra {arvores0} | {time.perf_counter() - t0:.1f} s")
print(f"limiar fora da dobra {r0['limiar']:.4f}")
for parte in ("treino", "validacao"):
    nomes = ("ce", "mse", "ce_corr", "mse_corr")
    print(parte, {k: round(float(np.mean([d[parte][k] for d in r0["perdas_dobras"]])), 4) for k in nomes})
print("plano, validação da dobra: KS 0,5294, 49 a 101 árvores")

KS médio 0.5294 | por dobra [0.5503 0.5519 0.5244 0.5059 0.5147]
árvores por dobra [101, 95, 49, 53, 92] | 5.4 s
limiar fora da dobra 0.4352
treino {'ce': 0.442, 'mse': 0.1453, 'ce_corr': 0.3783, 'mse_corr': 0.1218}
validacao {'ce': 0.4813, 'mse': 0.1609, 'ce_corr': 0.4194, 'mse_corr': 0.1361}
plano, validação da dobra: KS 0,5294, 49 a 101 árvores


## Estudo

Com `n_startup_trials=5`, o `MedianPruner` não corta nada nos 5 primeiros trials, então o corte não é exercitado neste estudo curto.
Ele é coberto em `tests/test_comum.py`.

In [5]:
pasta = comum.PASTA_RESULTADOS / "curto" if ESTUDO_CURTO else comum.PASTA_RESULTADOS
pasta.mkdir(parents=True, exist_ok=True)
arquivo = pasta / f"{MODELO}.db"
if ESTUDO_CURTO:
    arquivo.unlink(missing_ok=True)
armazenamento = f"sqlite:///{arquivo.as_posix()}"

# Mesma configuração de estudo() em modelos_jeff.py, até o criar_estudo do Diego entrar no comum.py.
optuna.logging.set_verbosity(optuna.logging.WARNING)
estudo = optuna.create_study(
    study_name=MODELO,
    storage=armazenamento,
    load_if_exists=True,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=comum.SEMENTE),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
)
if not estudo.trials:
    estudo.enqueue_trial(PADRAO)  # trial 0 = padrão da biblioteca

restantes = N_TRIALS - sum(t.state.is_finished() for t in estudo.trials)
t0 = time.perf_counter()
if restantes > 0:
    estudo.optimize(objetivo, n_trials=restantes)
tempo_parede = estudo.user_attrs.get("tempo_parede_s", 0) + time.perf_counter() - t0
estudo.set_user_attr("tempo_parede_s", tempo_parede)
print(f"{restantes} trials em {tempo_parede:.1f} s")

5 trials em 14.2 s


In [6]:
linhas = estudo.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs"))
colunas_tabela = {"number": "trial", "value": "ks", "state": "estado", "user_attrs_tempo_s": "tempo_s"}
tabela = linhas.rename(columns=colunas_tabela)
tabela["arvores"] = tabela["user_attrs_arvores_dobras"]
tabela = tabela.drop(columns=[c for c in tabela if c.startswith("user_attrs_")])
tabela.columns = [c.removeprefix("params_") for c in tabela.columns]
print(tabela.round(4).T.to_string(header=False))

trial                                      0                     1                    2                     3                          4
ks                                    0.5294                0.5258               0.4636                0.5034                     0.5035
estado                              COMPLETE              COMPLETE             COMPLETE              COMPLETE                   COMPLETE
learning_rate                            0.1                0.1206               0.2708                0.0801                     0.0117
loss                                log_loss           exponential             log_loss              log_loss                exponential
max_depth                                  3                     7                    9                     3                          7
max_features                             1.0                0.9063                0.513                0.8496                     0.9759
min_impurity_decrease                    

In [7]:
relido = optuna.load_study(study_name=MODELO, storage=armazenamento)
assert [t.value for t in relido.trials] == [t.value for t in estudo.trials]  # o SQLite guardou o estudo
assert relido.trials[0].params == PADRAO
assert relido.trials[0].value == r0["ks"]  # o trial 0 do estudo repete a medida acima

melhor = estudo.best_trial
print("melhor trial:", melhor.number, "KS", round(melhor.value, 4))
print("KS por dobra:", np.round(melhor.user_attrs["ks_dobras"], 4))
print("árvores por dobra:", melhor.user_attrs["arvores_dobras"])
print("n_estimators do treino final:", round(np.mean(melhor.user_attrs["arvores_dobras"])))

melhor trial: 0 KS 0.5294
KS por dobra: [0.5503 0.5519 0.5244 0.5059 0.5147]
árvores por dobra: [101, 95, 49, 53, 92]
n_estimators do treino final: 78


In [8]:
resumo = comum.resumir_estudo(estudo, N_TRIALS, tempo_parede)
for campo, valor in resumo.items():
    print(campo, "=", valor)

n_trials = 5
n_completos = 5
n_podados = 0
por_perda = {}
melhor_por_categoria = {'loss': {'log_loss': 0.5294474289697535, 'exponential': 0.5258072631407921}}
importancias = {'min_impurity_decrease': 0.23803558820591345, 'subsample': 0.23526248788218285, 'learning_rate': 0.1593672699745485, 'max_features': 0.12075419884453088, 'min_samples_split': 0.12000227319766771, 'min_samples_leaf': 0.09360258043320187, 'max_depth': 0.032496905490935826, 'loss': 0.00047869597101893253}
config_trial0 = {'loss': 'log_loss', 'learning_rate': 0.1, 'max_depth': 3, 'min_samples_split': 2, 'min_samples_leaf': 1, 'subsample': 1.0, 'max_features': 1.0, 'min_impurity_decrease': 0.0}
ks_trial0 = 0.5294474289697535
melhor_ks = 0.5294474289697535
convergencia = [0.5294474289697535, 0.5294474289697535, 0.5294474289697535, 0.5294474289697535, 0.5294474289697535]
tempo_parede_s = 14.18314499995904


## Próxima etapa

Ainda não executada, porque depende do N e usa o teste:

1. Estudo completo com `ESTUDO_CURTO = False` e o N definido.
2. Treino final por retreino: `final` do `carregar_dobras`, com o número de árvores do melhor trial e 5 sementes.
3. Gravação de `resultados/gradient_boosting.json` com `comum.salvar_resultado`.

---

# Roteiro original (21/10)

Mantido como registro.
A pré-proposta substituiu o protocolo de blocos pela busca conjunta no Optuna e tornou o XGBoost a referência da comparação, então as seções abaixo não descrevem o que este notebook faz.

## Papel de referência

Os outros notebooks medem a distância até a configuração aceita deste aqui, com a mesma métrica e as mesmas dobras. Enquanto um bloco deste experimento ainda não foi aceito, os outros usam o último bloco já aceito. Não usam um número provisório.


## Como o experimento avança

O experimento é uma sequência de blocos. Cada bloco faz uma intervenção só: um hiperparâmetro, uma escolha de arquitetura ou uma mudança nos dados. O bloco seguinte parte da configuração que venceu, e só fica no lugar dela se a métrica de validação subir.

A métrica é escolhida uma vez, antes do bloco 0 de qualquer modelo, e vale para os sete notebooks. Cerca de 26,5% dos clientes cancelam, então a métrica tem de enxergar a classe de churn. O conjunto de teste não entra nessa escolha. Ele só é usado no bloco final, com a configuração já congelada.

Empate fica com a configuração mais simples. Cada bloco aceito registra o que mudou, a configuração completa e a média e a dispersão nas cinco dobras.

O gradient boosting deste repositório é a referência de comparação. Cada modelo anota a distância do seu melhor bloco até o boosting congelado com o mesmo protocolo. O boosting também anda por blocos; a comparação usa a versão já aceita, sem perseguir um número que ainda está mudando.


## Dados

A entrada é a saída de `if702-preprocessamento.ipynb`: as cinco dobras estratificadas e a função `preparar` para o treino final e o teste. Em cada dobra o scaler é ajustado só no treino da dobra.

O oversampling repetitivo já está feito aí. O `RandomOverSampler` só duplica o treino da dobra. A validação e o teste continuam com um cliente por linha. Nenhum bloco refaz esse balanceamento por fora das dobras.

### Estratégias de dados compartilhadas

Nenhuma, além do pré-processamento.

Quando aparecer uma estratégia nova de exploração ou de mineração, ela é escrita nesta seção e copiada para os outros seis notebooks. Vira um bloco em cada experimento, em cima da configuração que aquele modelo já tiver aceito. O bloco mexe só nos dados. O modelo fica parado.


## Frentes já definidas

A biblioteca é escolhida no bloco 0 e fica congelada. Trocar de implementação no meio da sequência quebraria a comparação. Candidatas naturais são o gradient boosting do scikit-learn, o XGBoost e o LightGBM. O bloco 0 anota qual entrou e por qual restrição de ambiente.

A taxa de aprendizagem, o número de árvores, a profundidade e o número de folhas são as frentes de complexidade e de passo. Cada uma é um bloco. A taxa anda junto com o número de árvores: taxa menor pede mais rodadas. Por isso o número de árvores não é um teto escolhido no escuro. O early stopping na validação da dobra decide quantas árvores ficam, e a paciência desse critério é o hiperparâmetro a mais (`n_iter_no_change` no scikit-learn, `early_stopping_rounds` no XGBoost e no LightGBM).

Profundidade (`max_depth`) e número de folhas (`num_leaves`, `max_leaf_nodes` ou `max_leaves`) não entram no mesmo bloco. No LightGBM a folha é o controle principal de complexidade, e a documentação de ajuste pede que `num_leaves` fique abaixo de 2 elevado à profundidade. Uma árvore que cresce por folha, sem teto de profundidade, fica funda demais para o mesmo número de folhas.

Gini ou entropia mede a impureza do corte numa árvore de classificação e numa random forest. No gradient boosting a árvore ajusta o gradiente de uma função de erro, e o corte não usa Gini nem entropia. No scikit-learn 1.9 o parâmetro `criterion` do `GradientBoostingClassifier` está obsoleto e não muda o ajuste. A frente que ocupa esse lugar no boosting é a função de erro: `log_loss` ou `exponential`. A segunda recupera o AdaBoost e só serve para duas classes. Gini e entropia ficam para um bloco lateral, se ele for aberto, comparando uma árvore única ou uma random forest com o boosting. Esse bloco só substitui o boosting se a métrica vencer.

Além dessa lista, as documentações de ajuste do LightGBM e do XGBoost colocam mais quatro famílias, cada uma num bloco:

- Mínimo de amostra na folha. `min_data_in_leaf` no LightGBM, `min_samples_leaf` no scikit-learn, `min_child_weight` no XGBoost. É o freio mais citado quando a árvore cresce por folha. Se a biblioteca expuser amostra mínima e `min_child_weight`, são dois blocos.
- Subamostra de linhas. `subsample` ou `bagging_fraction`. No LightGBM só passa a valer com `bagging_freq` maior que zero; a frequência entra no mesmo bloco, porque sem ela a fração não age.
- Subamostra de colunas. `feature_fraction`, `colsample_bytree` ou `max_features`.
- Regularização do corte e da folha. `gamma` (`min_split_loss`) é a queda mínima da função de erro para aceitar um corte. `reg_lambda` é a penalidade L2 nos pesos das folhas e `reg_alpha` é a L1. No CatBoost a L2 se chama `l2_leaf_reg`. Se a biblioteca escolhida for o CatBoost, o esquema `Ordered` ou `Plain` é um bloco próprio.

O oversampling repetitivo é o mesmo das dobras, para a comparação ver a mesma amostra que as redes. Peso de classe (`class_weight`, `scale_pos_weight`) fica desligado enquanto esse oversampling estiver ativo. Ligar os dois no mesmo bloco mistura os efeitos.


## Hiperparâmetros que ainda vão chegar

Taxa, número de árvores com early stopping, profundidade, folhas, função de erro, amostra mínima na folha, subamostra de linhas, subamostra de colunas e regularização já estão nas frentes acima. O que ainda chegar entra como bloco novo, um item de cada vez, a partir da configuração vencedora.


## Ordem prevista dos blocos

1. Bloco 0. A biblioteca escolhida, no padrão dela, com early stopping já ligado numa paciência inicial, sobre as dobras com oversampling. Esse número já pode ser citado pelos outros notebooks.
2. Profundidade.
3. Número de folhas, abaixo de 2 elevado à profundidade aceita quando a biblioteca cresce por folha.
4. Mínimo de amostra na folha. Se a biblioteca também tiver `min_child_weight`, ele vem no bloco seguinte.
5. Taxa de aprendizagem. O número de árvores é reescolhido pelo early stopping, com a paciência ainda fixa.
6. Paciência do early stopping.
7. Subamostra de linhas, e a frequência de bagging se a biblioteca exigir.
8. Subamostra de colunas.
9. Queda mínima para aceitar um corte (`gamma`), se existir.
10. Regularização L2 das folhas, depois L1, se existir. No CatBoost, o esquema `Ordered` ou `Plain` entra antes da L2.
11. Função de erro do boosting: `log_loss` e `exponential`.
12. Bloco lateral, só se for aberto: Gini e entropia numa árvore única ou numa random forest, comparadas ao boosting aceito.
13. Itens que ainda chegarem.
14. Estratégias de dados compartilhadas, na mesma ordem dos outros experimentos.
15. Bloco final no teste. A partir daqui o número de referência para o relatório é o do teste, e os outros notebooks só comparam teste com teste.
